# Day 26: Agentic AI and Agentic RAG
**Agent components, the OpenAI Agents SDK, corrective RAG and multi-agent LangGraph**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas langgraph openai-agents nest_asyncio

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas langgraph openai-agents nest_asyncio
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. An agent from scratch: the components made visible

In [ ]:
import json, re
import genai_helpers

ORDERS = {"ORD-1042": {"status": "delivered", "amount": 4800, "days_late": 3},
          "ORD-2077": {"status": "in transit", "amount": 1200, "days_late": 0}}

def get_order_status(order_id: str) -> str:
    return json.dumps(ORDERS.get(order_id, {"error": "not found"}))

def calculate_refund(amount: float, days_late: int) -> str:
    return json.dumps({"refund": round(amount * min(0.05 * days_late, 0.5), 2)})

class MiniAgent:
    def __init__(self, instructions, tools, max_steps=5):
        self.instructions, self.tools, self.max_steps = instructions, tools, max_steps
        self.memory = []                                   # short-term memory: what happened in this task

    def plan(self, goal):
        # Offline planner: a stand-in for the model choosing the next action from memory.
        done = {m["tool"] for m in self.memory}
        oid = re.search(r"ORD-\d{4}", goal)
        if oid and "get_order_status" not in done:
            return ("get_order_status", {"order_id": oid.group(0)})
        last = json.loads(self.memory[-1]["result"]) if self.memory else {}
        if "refund" in goal.lower() and "amount" in last and "calculate_refund" not in done:
            return ("calculate_refund", {"amount": last["amount"], "days_late": last["days_late"]})
        return ("finish", None)

    def run(self, goal):
        for step in range(1, self.max_steps + 1):
            tool, args = self.plan(goal)
            if tool == "finish":
                if not self.memory:
                    return "Nothing to do."
                facts = "; ".join(m["result"] for m in self.memory)
                if genai_helpers.ONLINE:
                    return llm(f"Context: {facts}\nQuestion: {goal}", system=self.instructions)
                last = json.loads(self.memory[-1]["result"])       # offline: a simple templated answer
                return f"Your refund is {last['refund']} INR." if "refund" in last else f"Order details: {facts}"
            result = self.tools[tool](**args)
            self.memory.append({"step": step, "tool": tool, "args": args, "result": result})
            print(f"step {step}: {tool}({args}) -> {result}")
        return "Stopped: step limit reached"

agent = MiniAgent("You are a support agent. Answer from the context in one sentence.",
                  {"get_order_status": get_order_status, "calculate_refund": calculate_refund})
print("FINAL:", agent.run("ORD-1042 was late. What refund do I get?"))
print("Memory entries:", len(agent.memory))

## 3. The OpenAI Agents SDK (needs an API key)
The same idea with a real framework: an agent with a tool, plus a handoff to a specialist.

In [ ]:
if ONLINE:
    import nest_asyncio
    nest_asyncio.apply()                                    # notebooks already run an event loop
    from agents import Agent, Runner, function_tool

    @function_tool
    def order_status(order_id: str) -> str:
        """Look up an order by id, for example ORD-1042."""
        return get_order_status(order_id)

    @function_tool
    def refund_amount(amount: float, days_late: int) -> str:
        """Calculate a late-delivery refund."""
        return calculate_refund(amount, days_late)

    refunds_agent = Agent(name="Refunds", instructions="You handle refund questions. Use refund_amount. Be concise.",
                          tools=[order_status, refund_amount])
    triage = Agent(name="Triage", instructions="Answer order status questions with order_status. Hand refund questions to Refunds.",
                   tools=[order_status], handoffs=[refunds_agent])

    for q in ["Where is ORD-2077?", "ORD-1042 arrived 3 days late. What refund do I get?"]:
        result = Runner.run_sync(triage, q)
        print(f"Q: {q}\nA: {result.final_output}  (last agent: {result.last_agent.name})\n")
else:
    print("Offline: add OPENAI_API_KEY to run the Agents SDK. The MiniAgent above shows the same loop without a key.")

## 4. Corrective RAG with LangGraph

In [ ]:
%%writefile handbook.md
# Sakura Logistics Employee Handbook (sample)

## 1. Working hours
Standard working hours are 9:30 to 18:30, Monday to Friday. Teams in the Tokyo office follow JST and teams in the Bengaluru office follow IST.
Flexible start times between 8:00 and 11:00 are allowed with manager approval.

## 2. Leave policy
Full-time employees receive 18 days of paid leave per year. Leave accrues monthly at 1.5 days.
Unused leave can be carried forward up to a maximum of 10 days. Carried forward leave expires on 31 March of the following year.
Sick leave is separate: employees receive 8 days of paid sick leave per year. A medical certificate is required for sick leave longer than 2 consecutive days.

## 3. Remote work
Employees may work remotely up to 2 days per week. Remote work must be logged in the HR portal by 10:00 on the same day.
Remote work from another country requires approval from HR and the security team at least 14 days in advance.

## 4. Expenses
Travel expenses must be submitted within 30 days with receipts. Meals during business travel are reimbursed up to 2,000 INR or 4,000 JPY per day.
Expenses above 50,000 INR require approval from a director before booking.

## 5. Security
Laptops must use full disk encryption. Customer data must never be copied to personal devices or uploaded to unapproved AI tools.
Security incidents must be reported to security@sakura-logistics.example within 1 hour of discovery.

In [ ]:
import numpy as np
from typing import TypedDict
from langgraph.graph import StateGraph, END

text = open("handbook.md").read()
DOCS = [s.strip() for s in re.split(r"(?<=[.!?])\s+", re.sub(r"#+ .*\n", "", text)) if s.strip()]
DOC_VECS = embed(DOCS)
EXTERNAL = {"public holiday": "India and Japan public holidays are published by each office's HR team every December."}
SYNONYMS = {"roll over": "carried forward", "wfh": "remote work", "work from home": "remote work"}
WORDS = lambda t: {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in re.findall(r"[a-z0-9]+", t.lower()) if len(w) > 3}

class RagState(TypedDict, total=False):
    question: str
    docs: list
    answer: str
    rewrites: int
    path: list

def retrieve(s):
    sims = DOC_VECS @ embed(s["question"])[0]
    top = [DOCS[i] for i in np.argsort(sims)[::-1][:4]]
    return {"docs": top, "rewrites": s.get("rewrites", 0), "path": s.get("path", []) + ["retrieve"]}

def grade(s):
    if genai_helpers.ONLINE:
        keep = [d for d in s["docs"] if llm(f"Does this passage help answer '{s['question']}'? Reply yes or no.\n{d}", temperature=0).strip().lower().startswith("yes")]
    else:
        keep = [d for d in s["docs"] if len(WORDS(d) & WORDS(s["question"])) >= 2]   # offline stand-in grader
    return {"docs": keep, "path": s["path"] + [f"grade:{len(keep)}"]}

def rewrite(s):
    q = s["question"].lower()
    for k, v in SYNONYMS.items():
        q = q.replace(k, v)
    if genai_helpers.ONLINE:
        q = llm(f"Rewrite as a short search query for an employee handbook: {s['question']}", temperature=0)
    return {"question": q, "rewrites": s["rewrites"] + 1, "path": s["path"] + [f"rewrite:'{q}'"]}

def external(s):
    hits = [v for k, v in EXTERNAL.items() if k in s["question"].lower()]
    return {"docs": hits, "path": s["path"] + [f"external:{len(hits)}"]}

def generate(s):
    if not s["docs"]:
        return {"answer": "I don't know. Please contact HR.", "path": s["path"] + ["generate:none"]}
    ctx = "\n".join(s["docs"])
    return {"answer": llm(f"Context:\n{ctx}\n\nQuestion: {s['question']}", system="Answer only from the context."), "path": s["path"] + ["generate"]}

def decide(s):
    if s["docs"]:
        return "generate"
    return "rewrite" if s["rewrites"] < 1 else "external"

g = StateGraph(RagState)
for n, f in [("retrieve", retrieve), ("grade", grade), ("rewrite", rewrite), ("external", external), ("generate", generate)]:
    g.add_node(n, f)
g.set_entry_point("retrieve")
g.add_edge("retrieve", "grade")
g.add_conditional_edges("grade", decide, {"generate": "generate", "rewrite": "rewrite", "external": "external"})
g.add_edge("rewrite", "retrieve")
g.add_edge("external", "generate")
g.add_edge("generate", END)
crag = g.compile()

for q in ["How many days of sick leave do I get?", "How much leave can I roll over?", "Can I wfh from another country?", "When are the public holidays announced?"]:
    out = crag.invoke({"question": q})
    print(f"Q: {q}\n   path: {' -> '.join(out['path'])}\n   answer: {out['answer']}\n")

## 5. Multi-agent: a supervisor with specialists

In [ ]:
import pandas as pd

SALES = pd.DataFrame({"month": ["Sep", "Oct", "Nov"], "tickets": [820, 910, 1240], "bot_resolved": [0, 120, 460]})

class TeamState(TypedDict, total=False):
    task: str
    research: str
    analysis: str
    draft: str
    path: list

def supervisor(s):
    return {"path": s.get("path", []) + ["supervisor"]}

def route(s):
    for key, agent_name in [("research", "researcher"), ("analysis", "analyst"), ("draft", "writer")]:
        if not s.get(key):
            return agent_name
    return "done"

def researcher(s):
    hits = [d for d in DOCS if "ai tools" in d.lower() or "encryption" in d.lower()]
    return {"research": " ".join(hits[:2]) or "No policy found.", "path": s["path"] + ["researcher"]}

def analyst(s):
    df = SALES.assign(human_handled=SALES.tickets - SALES.bot_resolved)
    change = (df.tickets.iloc[-1] - df.tickets.iloc[0]) / df.tickets.iloc[0] * 100
    return {"analysis": f"Tickets rose {change:.0f}% from Sep to Nov; bot resolved {df.bot_resolved.iloc[-1]} in Nov; humans handled {df.human_handled.iloc[-1]}.",
            "path": s["path"] + ["analyst"]}

def writer(s):
    prompt = f"Context: {s['research']} {s['analysis']}\nQuestion: Write a 3 sentence update for the sponsor about: {s['task']}"
    draft = llm(prompt, system="Write clearly for a business sponsor.") if genai_helpers.ONLINE else f"Update: {s['analysis']} Policy reminder: {s['research'][:120]}"
    return {"draft": draft, "path": s["path"] + ["writer"]}

t = StateGraph(TeamState)
for n, f in [("supervisor", supervisor), ("researcher", researcher), ("analyst", analyst), ("writer", writer)]:
    t.add_node(n, f)
t.set_entry_point("supervisor")
t.add_conditional_edges("supervisor", route, {"researcher": "researcher", "analyst": "analyst", "writer": "writer", "done": END})
for n in ["researcher", "analyst", "writer"]:
    t.add_edge(n, "supervisor")
team = t.compile()

out = team.invoke({"task": "chatbot impact on support and any AI policy constraints"})
print("path:", " -> ".join(out["path"]))
print("\n" + out["draft"])

### TODO 1: Add a `checker` specialist that verifies the numbers in the draft match the analyst's results, and route to it before finishing.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)